# LitScope — Fine-Tuning Step 2: Fine-Tune SPECTER

Fine-tunes `allenai-specter` on IS-domain query–paper pairs using contrastive learning.
Then regenerates all paper embeddings with the new model.

**Pipeline:**
1. Baseline test — run sample queries against original SPECTER
2. Build training pairs from `synthetic_queries.csv`
3. Fine-tune with `MultipleNegativesRankingLoss`
4. Post-tuning test — same queries against fine-tuned SPECTER
5. Regenerate all embeddings and save

**Prerequisite:** Run `03_generate_queries.ipynb` first (need `synthetic_queries.csv`)

> **Runtime:** Use GPU (Runtime → Change runtime type → T4 GPU)

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_DIR  = '/content/drive/MyDrive/LitScope/data'
MODELS_DIR = '/content/drive/MyDrive/LitScope/models'
os.makedirs(MODELS_DIR, exist_ok=True)

CLASSIFIED_CSV     = f'{DRIVE_DIR}/papers_classified.csv'
QUERIES_CSV        = f'{DRIVE_DIR}/synthetic_queries.csv'
FINETUNED_MODEL    = f'{MODELS_DIR}/specter-is-finetuned'
EMBEDDINGS_FILE    = f'{DRIVE_DIR}/papers_embeddings.npy'

print(f'Model will be saved to: {FINETUNED_MODEL}')

## 2. Install Dependencies

In [ ]:
!pip install sentence-transformers datasets -q
import torch
print(f'PyTorch: {torch.__version__}')
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 3. Load Data

In [ ]:
import pandas as pd
import numpy as np

papers_df  = pd.read_csv(CLASSIFIED_CSV)
queries_df = pd.read_csv(QUERIES_CSV)

# Build doi → paper text lookup
paper_lookup = {
    row['doi']: f"{row['title']} [SEP] {str(row['abstract'])[:400]}"
    for _, row in papers_df.iterrows()
    if pd.notna(row['doi'])
}

print(f'Papers:  {len(papers_df)}')
print(f'Queries: {len(queries_df)} for {queries_df["doi"].nunique()} papers')
print(f'Coverage: {queries_df["doi"].nunique() / (papers_df["is_behavioral"]==True).sum() * 100:.1f}% of behavioral papers')

## 4. Baseline Test — Original SPECTER

**Core test (part 1/2):** Run 5 representative IS queries against the original SPECTER.
Record top-5 results. We will compare against fine-tuned results in Step 7.

In [ ]:
from sentence_transformers import SentenceTransformer

# Fixed test queries — covering different IS theories and contexts
TEST_QUERIES = [
    'social influence mobile payment adoption TAM UTAUT',
    'privacy concern information disclosure trust e-commerce',
    'technology acceptance perceived usefulness ease of use',
    'self-determination theory intrinsic motivation online learning',
    'protection motivation theory security behavior phishing',
]

print('Loading original SPECTER...')
base_model = SentenceTransformer('allenai-specter')

# Build baseline embeddings for all papers
print('Encoding all papers with baseline model...')
all_texts = [paper_lookup.get(row['doi'], f"{row['title']} [SEP] {str(row['abstract'])[:400]}")
             for _, row in papers_df.iterrows()]
base_embeddings = base_model.encode(all_texts, batch_size=64, show_progress_bar=True)
base_embeddings = base_embeddings.astype('float32')


def search_topk(query, embeddings, k=5):
    q_vec = base_model.encode(query).astype('float32')
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    q_norm = np.linalg.norm(q_vec)
    sims = (embeddings / (norms + 1e-9)) @ (q_vec / (q_norm + 1e-9))
    top_idx = np.argsort(sims)[::-1][:k]
    return [(papers_df.iloc[i]['title'], float(sims[i])) for i in top_idx]


print('\n' + '=' * 70)
print('  BASELINE RESULTS (original SPECTER)')
print('=' * 70)
baseline_results = {}
for q in TEST_QUERIES:
    results = search_topk(q, base_embeddings)
    baseline_results[q] = results
    print(f'\nQuery: "{q}"')
    for rank, (title, score) in enumerate(results, 1):
        print(f'  {rank}. [{score:.3f}] {title[:70]}')

## 5. Build Training Examples

Each `InputExample` is a (query, paper_text) pair.
With `MultipleNegativesRankingLoss`, all other papers in the batch serve as implicit negatives —
no need to manually select negative examples.

In [ ]:
from sentence_transformers import InputExample
from torch.utils.data import DataLoader

train_examples = []
skipped = 0

for _, row in queries_df.iterrows():
    doi = row['doi']
    if doi not in paper_lookup:
        skipped += 1
        continue
    train_examples.append(InputExample(texts=[row['query'], paper_lookup[doi]]))

print(f'Training examples: {len(train_examples)}')
print(f'Skipped (no DOI match): {skipped}')
print()
print('Sample training pair:')
ex = train_examples[0]
print(f'  Query: {ex.texts[0]}')
print(f'  Paper: {ex.texts[1][:100]}...')

## 6. Fine-Tune SPECTER

Hyperparameters:
- `lr = 2e-5` — small enough to preserve pre-trained knowledge
- `batch_size = 16` — larger batch = more implicit negatives per step
- `epochs = 3` — enough for this dataset size, avoids overfitting
- `warmup_ratio = 0.1` — protects pre-trained weights at the start

In [ ]:
from sentence_transformers import SentenceTransformer, losses

BATCH_SIZE   = 16
EPOCHS       = 3
LR           = 2e-5
WARMUP_RATIO = 0.1

print('Loading base SPECTER for fine-tuning...')
model = SentenceTransformer('allenai-specter')

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=BATCH_SIZE)
train_loss       = losses.MultipleNegativesRankingLoss(model)
warmup_steps     = int(len(train_dataloader) * EPOCHS * WARMUP_RATIO)

print(f'Training examples:  {len(train_examples)}')
print(f'Steps per epoch:    {len(train_dataloader)}')
print(f'Total epochs:       {EPOCHS}')
print(f'Warmup steps:       {warmup_steps}')
print(f'LR:                 {LR}')
print()

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=EPOCHS,
    warmup_steps=warmup_steps,
    optimizer_params={'lr': LR},
    show_progress_bar=True,
    output_path=FINETUNED_MODEL,
)

print(f'\nFine-tuning complete!')
print(f'Model saved: {FINETUNED_MODEL}')

## 7. Post-Tuning Test — Compare Before vs After

**Core test (part 2/2):** Run the same 5 queries against the fine-tuned model.
Compare top-5 results side-by-side with baseline.

In [ ]:
print('Loading fine-tuned model...')
ft_model = SentenceTransformer(FINETUNED_MODEL)

print('Encoding all papers with fine-tuned model...')
ft_embeddings = ft_model.encode(all_texts, batch_size=64, show_progress_bar=True).astype('float32')


def search_topk_ft(query, k=5):
    q_vec = ft_model.encode(query).astype('float32')
    norms = np.linalg.norm(ft_embeddings, axis=1, keepdims=True)
    q_norm = np.linalg.norm(q_vec)
    sims = (ft_embeddings / (norms + 1e-9)) @ (q_vec / (q_norm + 1e-9))
    top_idx = np.argsort(sims)[::-1][:k]
    return [(papers_df.iloc[i]['title'], float(sims[i])) for i in top_idx]


print('\n' + '=' * 70)
print('  BEFORE vs AFTER COMPARISON')
print('=' * 70)

for q in TEST_QUERIES:
    before = baseline_results[q]
    after  = search_topk_ft(q)
    print(f'\nQuery: "{q}"')
    print(f'  {"BEFORE":<38}  {"AFTER"}')
    print(f'  {"─"*36}  {"─"*36}')
    for i in range(5):
        b_title, b_score = before[i]
        a_title, a_score = after[i]
        print(f'  {i+1}. [{b_score:.3f}] {b_title[:30]:<30}  {i+1}. [{a_score:.3f}] {a_title[:30]}')

# Rank improvement metric: for each query, check if behavioral papers rank higher after tuning
print('\n' + '=' * 70)
print('  BEHAVIORAL PAPER RANK — Before vs After')
print('  (lower rank = appeared earlier in results = better)')
print('=' * 70)

behavioral_dois = set(papers_df[papers_df['is_behavioral'] == True]['doi'])

def avg_behavioral_rank(query, embeddings, encode_fn, topk=20):
    q_vec = encode_fn(query).astype('float32')
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    q_norm = np.linalg.norm(q_vec)
    sims = (embeddings / (norms + 1e-9)) @ (q_vec / (q_norm + 1e-9))
    top_idx = np.argsort(sims)[::-1][:topk]
    behavioral_ranks = [
        rank + 1 for rank, idx in enumerate(top_idx)
        if papers_df.iloc[idx]['doi'] in behavioral_dois
    ]
    return behavioral_ranks[:3]  # first 3 behavioral hits

for q in TEST_QUERIES:
    before_ranks = avg_behavioral_rank(q, base_embeddings, base_model.encode)
    after_ranks  = avg_behavioral_rank(q, ft_embeddings,   ft_model.encode)
    print(f'\nQuery: "{q}"')
    print(f'  Behavioral paper ranks — Before: {before_ranks}  After: {after_ranks}')

## 8. Regenerate All Embeddings with Fine-Tuned Model

In [ ]:
print('Regenerating embeddings for all papers...')
new_embeddings = ft_model.encode(all_texts, batch_size=64, show_progress_bar=True)
np.save(EMBEDDINGS_FILE, new_embeddings)

print(f'Saved: {EMBEDDINGS_FILE}')
print(f'Shape: {new_embeddings.shape}  (should be {len(papers_df)} × 768)')
print()
print('Next step: run 05_train_classifier.ipynb')